# Tutorial: snnTorch Integration

Seamless conversion between snnTorch spiking neural networks and TALON IR.

# 1. Define an snnTorch Model


In [ ]:
import torch
import torch.nn as nn
import snntorch as snn

class SimpleSNN(nn.Module):
    """Two-layer fully-connected SNN for MNIST classification."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.lif1 = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(128, 10)
        self.lif2 = snn.Leaky(beta=0.9)

    def forward(self, x, mem1=None, mem2=None):
        if mem1 is None:
            mem1 = self.lif1.init_leaky()
        if mem2 is None:
            mem2 = self.lif2.init_leaky()
        cur1 = self.fc1(x)
        spk1, mem1 = self.lif1(cur1, mem1)
        cur2 = self.fc2(spk1)
        spk2, mem2 = self.lif2(cur2, mem2)
        return spk2, mem1, mem2

model = SimpleSNN()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# 2. Export to TALON IR


In [ ]:
from snntorch.export_t1cir import export_to_ir
from talon import ir

sample = torch.randn(1, 784)
graph = export_to_ir(model, sample)

print(f"TALON IR graph:")
print(f"  Nodes: {len(graph.nodes)}")
print(f"  Edges: {len(graph.edges)}")
print()
for name, node in graph.nodes.items():
    print(f"  {name:8s} -> {type(node).__name__}")

# 3. Serialize to Disk


In [ ]:
import tempfile, os

path = os.path.join(tempfile.mkdtemp(), "snn_model.t1c")
ir.write(path, graph)

file_size = os.path.getsize(path)
print(f"Saved: {file_size:,} bytes")

loaded = ir.read(path)
print(f"Loaded: {len(loaded.nodes)} nodes, {len(loaded.edges)} edges")

# 4. Import Back to PyTorch


In [ ]:
from snntorch.import_t1cir import import_from_ir

executor = import_from_ir(path, return_state=True)

x = torch.randn(1, 784)
state = None

print("Multi-timestep inference:")
for t in range(10):
    out, state = executor(x, state)
    spike_count = (out > 0).sum().item()
    print(f"  t={t:2d}: output={out.shape}, spikes={spike_count}/10")

# 5. Round-Trip Verification


In [ ]:
original = export_to_ir(model, sample)
ir.write(path, original)
reimported = ir.read(path)

print("Round-trip verification:")
print(f"  Nodes match: {list(original.nodes.keys()) == list(reimported.nodes.keys())}")
print(f"  Edges match: {len(original.edges) == len(reimported.edges)}")

for name in original.nodes:
    orig_node = original.nodes[name]
    re_node = reimported.nodes[name]
    types_match = type(orig_node).__name__ == type(re_node).__name__
    print(f"  {name}: type={'ok' if types_match else 'MISMATCH'}")

# 6. Hybrid ANN-SNN Model


In [ ]:
class HybridSNN(nn.Module):
    """Conv feature extractor -> spiking classifier."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 14 * 14, 64)
        self.lif = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x, mem=None):
        if mem is None:
            mem = self.lif.init_leaky()
        x = self.pool(self.relu(self.conv1(x)))
        x = self.fc1(self.flatten(x))
        spk, mem = self.lif(x, mem)
        return self.fc2(spk), mem

hybrid = HybridSNN()
hybrid_graph = export_to_ir(hybrid, torch.randn(1, 1, 28, 28))

print(f"Hybrid ANN-SNN graph: {len(hybrid_graph.nodes)} nodes")
print()
for name, node in hybrid_graph.nodes.items():
    print(f"  {name:10s} -> {type(node).__name__}")

# 7. SDK Analysis & Profiling


In [ ]:
from talon import sdk

stats = sdk.analyze_graph(hybrid_graph)
print("=== Graph Analysis ===")
print(f"  Parameters:   {stats.total_params:,}")
print(f"  Memory:       {stats.total_bytes:,} bytes")
print(f"  FLOPs:        {stats.total_flops:,}")
print(f"  Depth:        {stats.depth}")
print(f"  Type counts:  {stats.type_counts}")

print()
prof = sdk.profile_graph(hybrid_graph)
print("=== Hardware Profile ===")
print(f"  Weight memory:     {prof.weight_memory:,} bytes")
print(f"  Activation memory: {prof.activation_memory:,} bytes")
print(f"  State memory:      {prof.state_memory:,} bytes")
print(f"  Total memory:      {prof.total_memory:,} bytes")
print(f"  MAC operations:    {prof.mac_ops:,}")
print(f"  Spike operations:  {prof.spike_ops:,}")
print(f"  Largest layer:     {prof.largest_layer}")

# 8. Linting & Fingerprinting


In [ ]:
lint_result = sdk.lint_graph(hybrid_graph)
print(f"Lint issues: {len(lint_result.issues)}")
for issue in lint_result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

print()
fp = sdk.fingerprint_graph(hybrid_graph)
print(f"Graph fingerprint: {fp[:32]}...")

# 9. Backend Simulation


In [ ]:
from talon.backend import get_backend

cpu = get_backend("cpu")

sim = cpu.simulate(hybrid_graph, n_steps=10)
print("=== CPU Simulation (10 steps) ===")
print(f"  Timesteps: {sim.timesteps_run}")
print(f"  Spike counts: {sim.spike_counts}")
print(f"  Output valid: {sim.outputs_valid}")

print()
prof_result = cpu.profile(hybrid_graph, n_steps=10, energy_preset="45nm_cmos")
print("=== Energy Profile (45nm CMOS) ===")
print(f"  Total latency:  {prof_result.total_latency_us:.1f} us")
print(f"  Total energy:   {prof_result.energy_estimate_uj:.4f} uJ")
print(f"    MAC energy:   {prof_result.mac_energy_uj:.4f} uJ")
print(f"    Spike energy: {prof_result.spike_energy_uj:.4f} uJ")
print(f"    SRAM energy:  {prof_result.sram_energy_uj:.4f} uJ")
print(f"  Peak memory:    {prof_result.peak_memory_bytes:,} bytes")